In [ ]:
import urllib.parse, urllib.request, time
import xml.etree.ElementTree as ET
from tqdm import tqdm

NS = {"a": "http://www.w3.org/2005/Atom"}
CAT = "(cat:math.* OR cat:stat.*)"
API = "http://export.arxiv.org/api/query"

# High-precision title phrases (dropped: `conjectures`, `selected problems`,
# `some problems`, `some questions` — too noisy; most hits were not open-problem lists).
title_phrases = [
    "open problems", "open problem",
    "open questions", "open question",
    "unsolved problems", "unsolved problem",
    "open conjectures",
    "problems and conjectures",
    "list of problems", "list of open problems",
    "problem list",
    "problem collection", "problem book",
    "collection of problems",
    "problem session", "problem sessions",
    "problem workshop",
    "research problems",
    "ten problems", "few problems",
]

def api_page(search_query, start, page_size=200):
    qs = urllib.parse.urlencode({
        "search_query": search_query,
        "start": start,
        "max_results": page_size,
        "sortBy": "submittedDate",
        "sortOrder": "descending",
    })
    with urllib.request.urlopen(f"{API}?{qs}", timeout=60) as r:
        return r.read().decode()

def fetch_phrase(phrase, page_size=200, max_pages=5, delay=3.5):
    q = f'ti:"{phrase}" AND {CAT}'
    out, start = [], 0
    for _ in range(max_pages):
        xml = api_page(q, start, page_size)
        root = ET.fromstring(xml)
        entries = root.findall("a:entry", NS)
        for e in entries:
            tid   = e.findtext("a:id", "", NS).strip()          # e.g. http://arxiv.org/abs/2504.04113v1
            title = " ".join(e.findtext("a:title", "", NS).split())
            abs_url = tid.replace("http://arxiv.org/abs/", "https://arxiv.org/abs/")
            abs_url = abs_url.rsplit("v", 1)[0] if "v" in abs_url.split("/")[-1] else abs_url
            out.append((abs_url, title))
        time.sleep(delay)
        if len(entries) < page_size:
            break
        start += page_size
    return out

rows = []
for phrase in tqdm(title_phrases):
    rows.extend(fetch_phrase(phrase))

# dedupe by link, keep first title seen
seen, links, titles = set(), [], []
for url, t in rows:
    if url in seen:
        continue
    seen.add(url)
    links.append(url)
    titles.append(t)

print(f"raw hits: {len(rows)}  unique links: {len(links)}")

In [12]:
links = list(set(links))

432

In [14]:
from tqdm import tqdm
titles = []
for link in tqdm(links):
    try:
        html = requests.get(link)
        bs4_obj = BeautifulSoup(html.text,'html.parser')
        title = bs4_obj.find('div',{'id':'content-inner'}).h1.text
        titles.append(title)
    except:
        continue

100%|█████████████████████████████████████████| 432/432 [02:27<00:00,  2.92it/s]


In [16]:
df = pd.DataFrame({'title':titles,'links':links})

In [17]:
df.sample(5)

,title,links
142,Title:Two open problems in the fixed point the...,https://arxiv.org/abs/1701.00519
330,Title:The solution of an open problem on semig...,https://arxiv.org/abs/2006.03180
202,Title:Non-Equivalence of Stochastic Optimal Co...,https://arxiv.org/abs/2012.13683
413,Title:On Brezis-Nirenberg problems: open quest...,https://arxiv.org/abs/2509.19863
204,Title:Numerical boundary flux functions that g...,https://arxiv.org/abs/2511.04197


In [ ]:
from litellm import completion
import os
# Set OPENAI_API_KEY in your environment before running this cell

from openai import OpenAI
client = OpenAI()

system_prompt = """
You will be given a link of an arxiv paper.
Im looking for papers that list open math problems. 
Your job is to open the link to validate whether the provided paper meets my expectations.
Return either True or False only.
"""

output = []
for _,row in tqdm(df.iterrows(),total=len(df)):
    response = client.responses.create(
        model="gpt-5-mini",
        tools=[{"type": "web_search"}],
        input=system_prompt + f"use this link {row.links}"
    )
    
    output.append(response.output_text)


In [26]:
df = pd.read_csv('arxiv-op-papers-0318_with_validation.csv')

In [36]:
df = df[~df.is_open_math_problem_list.str.contains('False')]

In [38]:
df.to_csv('Downloads/arxiv-op-papers-0318-gpt5mini-validated.csv', index=False)

In [39]:
import pandas as pd
df = pd.read_csv('Downloads/gpt_oss_120b_regenerate_5x5_first5.csv')

In [52]:
print(df.generation.values[0].split('assistantfinal')[1])

{
  "question_style": "proof",
  "difficulty": "research",
  "is_self_contained": true,
  "self_contained_explanation": "The problem statement defines all geometric objects (planar point set, triangulation, edge length, total weight) and the decision version of the optimization problem, so no external concepts are required beyond standard notions of polynomial time and NP‑completeness.",
  "subject_area": "Computational Geometry",
  "subfields": [
    "Geometric Algorithms",
    "Computational Complexity",
    "Combinatorial Optimization"
  ],
  "core_concepts": [
    "Triangulation of a planar point set",
    "Edge length and total weight of a triangulation",
    "Optimization (minimum total weight)",
    "Decision version of an optimization problem",
    "Polynomial‑time algorithms",
    "NP‑completeness and NP‑hardness"
  ],
  "required_theorems_or_results": [
    "Fundamental NP‑completeness theory (Cook‑Levin theorem, polynomial‑time reductions)",
    "Existence of triangulations 

In [ ]:
# === zbMATH title-search scraper ===
# Scrapes zbmath.org result pages for papers whose title matches open-problem phrases.
# zbMATH entries link to the source paper (often arxiv/journal), which we capture.
import requests, time, re
from bs4 import BeautifulSoup
from urllib.parse import quote_plus, urljoin
from tqdm import tqdm
import pandas as pd

ZB = "https://zbmath.org"
UA = {"User-Agent": "Mozilla/5.0 (open-problem-harvester)"}

zb_phrases = [
    "open problems", "open questions", "unsolved problems",
    "problem session", "problem collection", "problem list",
    "research problems", "problems and conjectures",
    "list of problems", "problem book",
]

def zb_fetch(phrase, max_pages=5, delay=2.5):
    rows, seen_entry = [], set()
    q = quote_plus(f'ti:"{phrase}"')
    for page in range(1, max_pages + 1):
        url = f"{ZB}/?q={q}&p={page}"
        try:
            r = requests.get(url, timeout=60, headers=UA)
        except Exception as e:
            print(f"[zb] {phrase} p{page}: {e}")
            break
        if r.status_code != 200:
            print(f"[zb] {phrase} p{page}: HTTP {r.status_code}")
            break
        soup = BeautifulSoup(r.text, "html.parser")
        # Each result is a block; pull the Zbl entry link + visible title + any external link (arxiv/DOI)
        items = soup.select("div.list > div, div.results-list div.item")
        if not items:
            # fallback: walk all anchors to entry pages
            items = [soup]
        hit = 0
        for it in items:
            a_entry = it.find("a", href=re.compile(r"q=an(%3A|:)"))
            if not a_entry:
                continue
            entry = urljoin(ZB, a_entry["href"])
            if entry in seen_entry:
                continue
            seen_entry.add(entry)
            title = " ".join(a_entry.get_text(" ").split())
            # external links inside the same item (arxiv / doi / publisher)
            ext = []
            for a in it.find_all("a", href=True):
                h = a["href"]
                if "arxiv.org" in h or "doi.org" in h:
                    ext.append(h)
            rows.append({"entry": entry, "title": title, "external": "|".join(ext), "phrase": phrase})
            hit += 1
        if hit == 0:
            break
        time.sleep(delay)
    return rows

zb_rows = []
for p in tqdm(zb_phrases):
    zb_rows.extend(zb_fetch(p))

zb_df = pd.DataFrame(zb_rows).drop_duplicates("entry") if zb_rows else pd.DataFrame(columns=["entry","title","external","phrase"])
print(f"zbmath unique entries: {len(zb_df)}")
zb_df.head(10)


In [ ]:
# === Google Search harvester ===
# Uses googlesearch-python (pip install googlesearch-python).
# Queries target workshop/survey PDFs and open-problem lists.
from googlesearch import search
import time
from tqdm import tqdm
import pandas as pd

google_queries = [
    "open problems workshop pdf",
    "list of open problems mathematics pdf",
    "unsolved problems survey pdf",
    "problem session proceedings pdf",
    "open problems in algebra pdf",
    "open problems in topology pdf",
    "open problems in number theory pdf",
    "open problems in combinatorics pdf",
    "open problems in analysis pdf",
    "open problems in probability pdf",
    "open problems in geometry pdf",
    "open problems in graph theory pdf",
    "open questions mathematics survey filetype:pdf",
    'site:arxiv.org "open problems" survey',
]

g_rows = []
for q in tqdm(google_queries):
    try:
        for url in search(q, num_results=30, sleep_interval=2, lang="en"):
            g_rows.append({"query": q, "url": url})
    except Exception as e:
        print(f"[google] {q}: {e}")
    time.sleep(5)

g_df = pd.DataFrame(g_rows).drop_duplicates("url") if g_rows else pd.DataFrame(columns=["query","url"])
print(f"google unique urls: {len(g_df)}")
g_df.head(10)
